# 02 · Popularity-based recommender

Non-personalised baseline: every user gets the same list (minus movies they already rated). Three scoring rules:

* **count** — number of ratings $n_i$;
* **mean** — average rating $\bar r_i$;
* **bayesian** — damped mean $\tilde r_i = \dfrac{\sum_u r_{ui} + m\,\mu}{n_i + m}$, which shrinks the mean of rarely rated movies towards the global mean $\mu$ and so uses both the average and the number of ratings. $m$ acts as a prior of $m$ “virtual” ratings equal to $\mu$.

For rating prediction (RMSE) the model predicts $\tilde r_i$.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)

def save_params(name, params):
    path = RESULTS / "best_params.json"
    allp = json.loads(path.read_text()) if path.exists() else {}
    allp[name] = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
    path.write_text(json.dumps(allp, indent=2))

In [ ]:
from src.data import load_ratings, load_movies, temporal_user_split
ratings, movies = load_ratings(), load_movies()
train, test = temporal_user_split(ratings, test_frac=0.2)
train_in, valid = temporal_user_split(train, test_frac=0.2)
title = movies.set_index("item_idx")["title"]
print(f"train={len(train)} test={len(test)} | train_in={len(train_in)} valid={len(valid)}")

In [ ]:
from src.models import PopularityRecommender
from src.evaluation import grid_search, evaluate_ranking, evaluate_rmse
tops = {}
for method in ["count", "mean", "bayesian"]:
    m = PopularityRecommender(method=method, m=20).fit(train)
    tops[method] = title[np.argsort(-m.item_scores_)[:10]].values
pd.DataFrame(tops, index=range(1, 11))

The raw mean surfaces obscure movies rated 5 by one or two people; count and the damped mean give well-known titles.

## Hyper-parameter selection on the validation split

`m` only affects the `bayesian` rule, so the `count` and `mean` rows repeat.

In [ ]:
grid_rank = grid_search(PopularityRecommender, {"method": ["count", "mean", "bayesian"], "m": [5, 20, 50, 100]},
                        train_in, valid, objective="ndcg@10", verbose=False)
grid_rank

In [ ]:
grid_rmse = grid_search(lambda m: PopularityRecommender("bayesian", m), {"m": [0, 1, 5, 10, 20, 50, 100]},
                        train_in, valid, objective="rmse", verbose=False)
grid_rmse

In [ ]:
best_rank = grid_rank.iloc[0][["method", "m"]].to_dict()
best_rmse = {"method": "bayesian", "m": grid_rmse.iloc[0]["m"]}
save_params("popularity_ranking", best_rank); save_params("popularity_rating", best_rmse)
grid_rank.to_csv(RESULTS / "grid_popularity_ranking.csv", index=False)
grid_rmse.to_csv(RESULTS / "grid_popularity_rmse.csv", index=False)
best_rank, best_rmse

## Test-set performance (selected configuration, refit on the full training set)

In [ ]:
rank_model = PopularityRecommender(**best_rank).fit(train)
rate_model = PopularityRecommender(**best_rmse).fit(train)
res = evaluate_ranking(rank_model, train, test, ks=(10,))
res["rmse"] = evaluate_rmse(rate_model, test)
pd.Series(res)